In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!ls

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PySparkLabProject").getOrCreate()

In [ ]:
employees_df = spark.read.csv("employees.csv", header=True, inferSchema=True)

departments_df = spark.read.csv("departments.csv", header=True, inferSchema=True)

sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)

nested_df = spark.read.option("multiline","true").json("employees_nested.json")

In [ ]:
employees_df.show()
departments_df.show()
sales_df.show()
nested_df.show()

In [ ]:
employees_df.printSchema()
departments_df.printSchema()
sales_df.printSchema()
nested_df.printSchema()

In [ ]:
employees_df.show(5)

In [ ]:
employees_df.count()

In [ ]:
employees_df.columns

In [ ]:
employees_df.show(3)

In [ ]:
employees_df.select("name", "salary").show()

In [ ]:
employees_df.withColumnRenamed("salary", "emp_salary").show()

In [ ]:
employees_df.filter(employees_df.joining_date > "2023-01-01").show()

In [ ]:
from pyspark.sql.functions import col


In [ ]:
employees_df.filter(col("salary") > 65000).show()

In [ ]:
employees_df.filter(col("department") == "IT").show()

In [ ]:
employees_df.filter(
    (col("department") == "IT") & (col("salary") > 70000)
).show()

In [ ]:
employees_df.drop("joining_date").show()

In [ ]:
employees_df.orderBy("salary").show()

In [ ]:
employees_df.orderBy(col("salary").desc()).show()

In [ ]:
employees_df.orderBy(col("salary").desc()).show(3)

In [ ]:
from pyspark.sql.functions import sum, avg, max, min

employees_df.select(sum("salary")).show()
employees_df.select(avg("salary")).show()
employees_df.select(max("salary")).show()
employees_df.select(min("salary")).show()

In [ ]:
employees_df.groupBy("department").count().show()

employees_df.groupBy("department").avg("salary").show()

employees_df.groupBy("department").sum("salary").show()

In [ ]:
employees_df.join(departments_df, "department").show()

employees_df.join(departments_df, "department", "left").show()

employees_df.join(departments_df, "department") \
    .select("name", "location") \
    .show()

In [ ]:
from pyspark.sql.functions import col, lit, upper, length, when, to_date, year, month, explode, sum, avg

In [ ]:
employees_df.withColumn("bonus", col("salary") * 0.10).show()

In [ ]:
employees_df.withColumn("company", lit("BotCampus")).show()

In [ ]:
employees_df.filter(col("department").isNull()).show()

In [ ]:
employees_df.fillna({"department": "Unknown"}).show()

In [ ]:
employees_df.dropna().show()

In [ ]:
employees_df.withColumn("name_upper", upper(col("name"))).show()

In [ ]:
employees_df.withColumn("name_length", length(col("name"))).show()

In [ ]:
employees_df.withColumn("salary_in_lakhs", col("salary") / 100000).show()

In [ ]:
employees_df.withColumn(
    "salary_grade",
    when(col("salary") >= 75000, "High")
    .when((col("salary") >= 60000) & (col("salary") <= 74999), "Medium")
    .otherwise("Low")
).show()

In [ ]:
employees_date_df = employees_df.withColumn("joining_date", to_date(col("joining_date"), "yyyy-MM-dd"))
employees_date_df.show()

In [ ]:
employees_date_df.withColumn("joining_year", year(col("joining_date"))) \
                 .withColumn("joining_month", month(col("joining_date"))) \
                 .show()

In [ ]:
new_data = [
    (8, "Divya", "Finance", 67000, "2023-05-01"),
    (9, "Vikram", "IT", 73000, "2023-06-15")
]

new_columns = ["emp_id", "name", "department", "salary", "joining_date"]

new_employees_df = spark.createDataFrame(new_data, new_columns)
new_employees_df.show()

In [ ]:
employees_df.union(new_employees_df).show()

In [ ]:
employees_df.unionByName(new_employees_df).show()

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, row_number

In [ ]:
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

employees_df.withColumn("rank", rank().over(window_spec)).show()

In [ ]:
employees_df.withColumn("row_number", row_number().over(window_spec)).show()

In [ ]:
nested_df = spark.read.option("multiline", "true").json("employees_nested.json")
nested_df.show(truncate=False)
nested_df.printSchema()

In [ ]:
nested_df.select(
    "emp_id",
    col("address.city").alias("city"),
    col("address.state").alias("state")
).show()

In [ ]:
nested_df.select("name", explode(col("skills")).alias("skill")).show()

In [ ]:
sales_df.select(sum("amount").alias("total_sales")).show()

In [ ]:
sales_df.groupBy("emp_id").agg(sum("amount").alias("total_sales")).show()

In [ ]:
sales_df.groupBy("emp_id") \
       .agg(sum("amount").alias("total_sales")) \
       .orderBy(col("total_sales").desc()) \
       .show(1)

In [ ]:
sales_df.groupBy("product").agg(sum("amount").alias("total_sales")).show()

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [ ]:
def categorize_sales(amount):
    if amount > 50000:
        return "Premium"
    elif 20000 <= amount <= 50000:
        return "Medium"
    else:
        return "Small"

sales_category_udf = udf(categorize_sales, StringType())

In [ ]:
employees_pandas = employees_df.toPandas()
employees_pandas

In [ ]:
employees_pandas[employees_pandas["salary"] > 65000]

In [ ]:
import ipywidgets as widgets
from IPython.display import display

In [ ]:
department_dropdown = widgets.Dropdown(
    options=["All", "IT", "HR", "Finance"],
    value="All",
    description="Department:"
)

display(department_dropdown)

In [ ]:
salary_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=100000,
    step=1000,
    description="Min Salary:"
)

display(salary_slider)

In [ ]:
def filter_employees(department, min_salary):
    filtered_df = employees_df

    if department != "All":
        filtered_df = filtered_df.filter(col("department") == department)

    filtered_df = filtered_df.filter(col("salary") >= min_salary)
    filtered_df.show()

In [ ]:
widgets.interactive(filter_employees, department=department_dropdown, min_salary=salary_slider)

In [ ]:
#Final Mini Project

from pyspark.sql.functions import col, avg, sum

# Top Employee
top_employee = employees_df.orderBy(col("salary").desc()).first()["name"]

# Average Salary
average_salary = employees_df.select(avg("salary").alias("avg_salary")).first()["avg_salary"]

# Department Employee Count
department_counts = employees_df.groupBy("department").count().collect()

# Total Sales
total_sales = sales_df.select(sum("amount").alias("total_sales")).first()["total_sales"]

# Best Sales Employee
best_sales_row = sales_df.groupBy("emp_id") \
                         .agg(sum("amount").alias("total_sales")) \
                         .orderBy(col("total_sales").desc()) \
                         .first()

best_emp_id = best_sales_row["emp_id"]
best_sales_employee = employees_df.filter(col("emp_id") == best_emp_id).select("name").first()["name"]

# Create report text
report_text = f"""FINAL REPORT
Top Employee: {top_employee}
Average Salary: {average_salary}

Department Employee Count:
"""

for row in department_counts:
    report_text += f"{row['department']}: {row['count']}\n"

report_text += f"""
Total Sales: {total_sales}
Best Sales Employee: {best_sales_employee}
"""

print(report_text)

# Save to final_report.txt
with open("final_report.txt", "w") as file:
    file.write(report_text)

print("final_report.txt saved successfully.")

